# Titanic Survival Prediction using Logistic Regression

## Project Overview

This project predicts whether a passenger survived the Titanic disaster using machine learning. The dataset is taken from Kaggle's Titanic competition.

The workflow includes:

* Data loading and exploration
* Handling missing values
* Feature encoding
* Feature scaling using StandardScaler
* Training a Logistic Regression model
* Evaluating model performance on a validation set

### Final Results

* Training Accuracy: ~80%
* Validation Accuracy: ~79.2%

### Future Improvements

* Feature engineering
* XGBoost classifier
* Hyperparameter tuning
* Better utilization of Cabin information


##  #Import Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Load Dataset

In [2]:
df_train=pd.read_csv('/kaggle/input/datasets/abeermalviya/titanic1stcompetition/train.csv')
df_test=pd.read_csv('/kaggle/input/datasets/abeermalviya/titanic1stcompetition/test.csv')

df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Exploratory Data Analysis

In [3]:
df_train.shape

(891, 12)

In [4]:
df_train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## Missing Values Analysis

In [5]:
df_train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Handling Missing Values

The Cabin column contains 687 missing values out of 891 observations. Due to the large proportion of missing data, the column was removed from the baseline model. Missing values in Age were replaced with the mean age, and the two missing Embarked values were removed.

In [6]:
# Fill missing ages with mean age
df_train['Age'] = df_train['Age'].fillna(df_train['Age'].mean())

# Drop Cabin due to excessive missing values
df_train.drop('Cabin',axis=1, inplace=True)

# Remove rows with missing embarked values
df_train.dropna(subset=['Embarked'],inplace=True)

## Verify Missing Value Handling

In [7]:
df_train.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

## Feature Selection
The Name and Ticket columns were removed from the baseline model because they are textual identifiers and cannot be directly used by Logistic Regression without additional feature engineering.

In [8]:
df_train.drop(['Name', 'Ticket'], axis=1, inplace=True)

## Feature Encoding

Machine learning models require numerical input features. Therefore, categorical variables were converted into numerical representations before training the Logistic Regression model.

In [9]:
# Encode Sex column
df_train['Sex'] = df_train['Sex'].map({
    'male': 1,
    'female': 0
})

### One-Hot Encoding Embarked
The Embarked column contains three categories (S, C, and Q). One-hot encoding was used to convert these categories into numerical features suitable for Logistic Regression.

In [10]:
embarked_dummies=pd.get_dummies(df_train['Embarked'],prefix='Embarked')

df_train=pd.concat([df_train, embarked_dummies], axis=1)

df_train.drop('Embarked', axis=1, inplace=True)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [11]:
df_train[['Embarked_C', 'Embarked_Q', 'Embarked_S']] = (
    df_train[['Embarked_C', 'Embarked_Q', 'Embarked_S']].astype(int)
)

## Final Feature Cleanup

PassengerId is a unique identifier and does not contain meaningful information about passenger survival. Therefore, it was removed before training the model.

In [30]:
df_train.drop('PassengerId', axis=1, inplace=True)

## Preparing Features and Target Variable
The target variable is Survived. All remaining columns are used as input features for training the model.

In [12]:
y=df_train['Survived']
x=df_train.drop(['Survived'],axis=1)

## Train-Test Split
The dataset was split into training and testing sets using an 80-20 ratio. The training set was used for model fitting, while the test set was used to evaluate generalization performance.

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=2)

## Feature Scaling
Standardization (Z-score normalization) was applied to the training and test features. The scaler was fitted only on the training data to prevent data leakage.

In [15]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## Model Training

A Logistic Regression model was trained on the standardized training data.

In [17]:
model=LogisticRegression()

In [18]:
model.fit(x_train,y_train)

LogisticRegression()

## Model Evaluation
The model achieved approximately 80.7% accuracy on the training set.

In [19]:
training_data_prediction=model.predict(x_train)

In [20]:
training_accuracy = accuracy_score(y_train, training_data_prediction)
print(f"Training Accuracy: {training_accuracy:.4f}")

Training Accuracy: 0.8073


In [21]:
test_data_prediction=model.predict(x_test)

In [22]:
test_accuracy=accuracy_score(y_test,test_data_prediction)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.7921


## Conclusion

The Logistic Regression model achieved a training accuracy of approximately **80.7%** and a test accuracy of approximately **79.2%**. The small gap between training and test performance suggests that the model generalizes reasonably well and is not significantly overfitting.

While the baseline model produced solid results, there is still room for improvement through additional feature engineering, hyperparameter tuning, and experimenting with more advanced models such as XGBoost. These enhancements will be explored in future iterations of the project.
